# Beispiele: Dimensionsreduktion

PCA mit erklärter Streuung, t-SNE und UMAP. Datensätze: `load_wine` (13 Messwerte) und `load_digits` (8 × 8 Pixel).

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/05_Dimensionsreduktion.ipynb) (beim ersten Mal mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

**Mehr dazu in eigenen Notebooks:**
- [PCA_gewichtete_Summen_own.ipynb](../PCA_gewichtete_Summen_own.ipynb) ([in Colab](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/PCA_gewichtete_Summen_own.ipynb)): was in einer Hauptkomponente steckt, Schritt für Schritt von Hand nachgerechnet

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
Xw, yw = load_wine(return_X_y=True, as_frame=True)
Xw_z = StandardScaler().fit_transform(Xw)   # PCA needs scaled features

## PCA

Wie viele Hauptkomponenten braucht es für 95 % der Streuung? Danach die 13 Wein-Merkmale in 2D.

In [ ]:
pca_voll = PCA().fit(Xw_z)
kumuliert = np.cumsum(pca_voll.explained_variance_ratio_)
print("Komponenten für 95 %:", int(np.argmax(kumuliert >= 0.95) + 1), "von", Xw.shape[1])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
a1.plot(range(1, 14), kumuliert, "o-"); a1.axhline(0.95, ls="--", c="grey"); a1.set_title("Erklärte Streuung (kumuliert)")
Z = PCA(n_components=2).fit_transform(Xw_z)
a2.scatter(Z[:, 0], Z[:, 1], c=yw, s=10, cmap="tab10"); a2.set_title("Weine in 2D (Farbe = Sorte)")
plt.tight_layout(); plt.show()

## t-SNE und UMAP

Handgeschriebene Ziffern: 8 × 8 Pixel = 64 Merkmale. PCA (linear) gegen t-SNE und UMAP (nicht linear). Die Farbe ist die echte Ziffer und dient nur zur Kontrolle.

In [ ]:
from sklearn.manifold import TSNE

Xdig, ydig = load_digits(return_X_y=True)
einbettungen = {"PCA": PCA(n_components=2).fit_transform(Xdig),
                "t-SNE": TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(Xdig)}
try:
    import umap
    einbettungen["UMAP"] = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(Xdig)
except ImportError:
    print("umap fehlt: pip install umap-learn")

fig, axes = plt.subplots(1, len(einbettungen), figsize=(4 * len(einbettungen), 3.5))
for ax, (name, E) in zip(axes, einbettungen.items()):
    ax.scatter(E[:, 0], E[:, 1], c=ydig, s=3, cmap="tab10"); ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Deutung:** PCA lässt die Ziffern stark überlappen. t-SNE und UMAP trennen sie klar. Die Abstände **zwischen** den Gruppen und deren Grösse bedeuten dort aber nichts. Die Bilder dienen nur zum Anschauen, nicht zum Weiterrechnen.